# Travel Hotel Reviews RAG Assistant

A **Retrieval-Augmented Generation (RAG)** pipeline that answers natural-language questions about guest reviews using Databricks AI Search and Foundation Models.

---

## How RAG Works in This Notebook

Traditional LLMs generate answers from their pre-trained knowledge, which can be outdated, vague, or disconnected from your real business data. **RAG** solves this by grounding the LLM's answer in your actual data — here, guest reviews stored in a Databricks AI Search index.

The pipeline has two stages:

### 1. Retrieve — Semantic + Keyword Search (`HYBRID`)
The user's question is sent as a query to the `workspace.wanderbricks_rag.review_index` AI Search index. This index was built from a Delta table of guest reviews, where Databricks automatically computed embedding vectors for each review's `comment` text using a built-in embedding model. At query time, the index performs a **HYBRID search** that combines:

* **Semantic similarity** — compares the meaning of the question against the embedding vectors of all reviews to find conceptually related content (even if exact words differ)
* **Keyword matching** — uses BM25 scoring to ensure exact terms like property names, locations, or amenities are surfaced

The top 5 most relevant reviews are returned along with a similarity score for each.

### 2. Generate — Foundation Model Answer
The retrieved reviews are formatted into a context prompt and sent to the `databricks-meta-llama-3-3-70b-instruct` Foundation Model. The LLM is instructed to answer **only** based on the provided reviews — not its own training data — and to cite specific examples. This ensures every answer is grounded, traceable, and explainable.

### Why This Matters for Travel Hotel Reviews

| Without RAG | With RAG (this notebook) |
| --- | --- |
| LLM guesses based on generic knowledge | Answers from actual guest review data |
| No source citations | Each answer links back to specific review IDs & scores |
| Cannot filter to your latest data | Always queries the live AI Search index (synced from Delta) |
| Hallucinations about property-specific details | Grounded in real review text with ratings & host info |

---

## Data Source

* **AI Search Index**: `workspace.wanderbricks_rag.review_index`
* **Endpoint**: `reviews_rag_ep`
* **Primary Key**: `review_id`
* **Indexed Columns**: `comment`, `property_title`, `rating`, `host_name`, `property_description`, `property_type`, `base_price`, and more
* **Embedding**: 1024-dimensional vectors auto-computed by Databricks on the `comment` column

## Execution Steps

1. **Run Cell 2** — Connects to the Databricks workspace and initializes the AI Search client
2. **Run Cell 3** — Queries the index with a sample question and displays retrieved reviews
3. **Run Cell 4** — Passes retrieved reviews to the LLM and prints a grounded answer
4. **Run Cell 5** — Calls the interactive `ask_review_question()` function with 3 demo questions

> **Tip**: Modify the questions in Cell 5 or pass your own to `ask_review_question("your question here")` to explore different aspects of the review data.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

# --- Configuration ---
ENDPOINT_NAME = "reviews_rag_ep"
INDEX_NAME = "workspace.wanderbricks_rag.review_index"
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
NUM_RESULTS = 5

# Create the Databricks workspace client (auto-authenticated in-notebook)
w = WorkspaceClient()

print(f"✓ Connected to workspace. AI Search endpoint: {ENDPOINT_NAME}")
print(f"  Index: {INDEX_NAME}")
print(f"  LLM:   {LLM_ENDPOINT}")

In [0]:
# Define a sample user question
user_question = "What do customers think about room service?"

# Query the index using HYBRID search (semantic + keyword)
query_result = w.vector_search_indexes.query_index(
    index_name=INDEX_NAME,
    columns=["review_id", "comment", "property_title", "rating", "host_name"],
    query_text=user_question,
    query_type="HYBRID",
    num_results=NUM_RESULTS,
)

# Display retrieved reviews
print(f"Question: {user_question}\n")
print(f"Retrieved {len(query_result.result.data_array)} reviews:\n")
print("-" * 80)

for i, row in enumerate(query_result.result.data_array, 1):
    review_id = row[0]
    comment = row[1]
    property_title = row[2]
    rating = row[3]
    host_name = row[4]
    score = row[-1]  # similarity score is always the last element

    print(f"  [{i}] Review ID: {review_id} | Score: {score:.4f}")
    print(f"      Property: {property_title} | Rating: {rating} | Host: {host_name}")
    print(f"      Comment:  {comment}")
    print()

# Save results for the next cell
retrieved_reviews = query_result.result.data_array

In [0]:
# Build context from retrieved reviews
context_parts = []
for i, row in enumerate(retrieved_reviews, 1):
    review_id = row[0]
    comment = row[1]
    property_title = row[2]
    rating = row[3]
    host_name = row[4]
    context_parts.append(
        f"Review {i} (ID: {review_id}, Property: {property_title}, "
        f"Rating: {rating}, Host: {host_name}):\n{comment}"
    )

context_text = "\n\n".join(context_parts)

# Construct the RAG prompt
system_prompt = (
    "You are a helpful assistant analyzing guest reviews for a travel platform. "
    "Answer the user's question based ONLY on the provided reviews. "
    "If the reviews don't contain enough information, say so. "
    "Summarize the key sentiments and provide specific examples from the reviews."
)

user_prompt = f"Guest reviews:\n\n{context_text}\n\nQuestion: {user_question}"

# Call the Foundation Model
llm_response = w.serving_endpoints.query(
    name=LLM_ENDPOINT,
    messages=[
        ChatMessage(role=ChatMessageRole.SYSTEM, content=system_prompt),
        ChatMessage(role=ChatMessageRole.USER, content=user_prompt),
    ],
    max_tokens=512,
    temperature=0.3,
)

answer = llm_response.choices[0].message.content

print("=" * 80)
print("QUESTION:")
print(f"  {user_question}")
print("=" * 80)
print("\nANSWER:")
print(answer)
print("\n" + "=" * 80)
print(f"Based on {len(retrieved_reviews)} retrieved reviews.")

In [0]:
def ask_review_question(question: str, num_results: int = 5):
    """
    RAG pipeline: retrieve relevant reviews from the AI Search index,
    then generate an answer using a Foundation Model.

    Args:
        question: A natural-language question about guest reviews.
        num_results: Number of reviews to retrieve (default 5).

    Returns:
        A dict with 'answer' (str) and 'sources' (list of dicts with review details).
    """
    # --- Step 1: Retrieve relevant reviews via HYBRID search ---
    query_result = w.vector_search_indexes.query_index(
        index_name=INDEX_NAME,
        columns=["review_id", "comment", "property_title", "rating", "host_name"],
        query_text=question,
        query_type="HYBRID",
        num_results=num_results,
    )

    # Extract review data
    sources = []
    context_parts = []
    for i, row in enumerate(query_result.result.data_array, 1):
        review_id = row[0]
        comment = row[1]
        property_title = row[2]
        rating = row[3]
        host_name = row[4]
        score = row[-1]

        sources.append({
            "review_id": review_id,
            "comment": comment,
            "property_title": property_title,
            "rating": rating,
            "host_name": host_name,
            "score": score,
        })
        context_parts.append(
            f"Review {i} (ID: {review_id}, Property: {property_title}, "
            f"Rating: {rating}, Host: {host_name}):\n{comment}"
        )

    context_text = "\n\n".join(context_parts)

    # --- Step 2: Generate answer with Foundation Model ---
    system_prompt = (
        "You are a helpful assistant analyzing guest reviews for a travel platform. "
        "Answer the user's question based ONLY on the provided reviews. "
        "If the reviews don't contain enough information, say so. "
        "Summarize the key sentiments and provide specific examples from the reviews."
    )
    user_prompt = f"Guest reviews:\n\n{context_text}\n\nQuestion: {question}"

    llm_response = w.serving_endpoints.query(
        name=LLM_ENDPOINT,
        messages=[
            ChatMessage(role=ChatMessageRole.SYSTEM, content=system_prompt),
            ChatMessage(role=ChatMessageRole.USER, content=user_prompt),
        ],
        max_tokens=512,
        temperature=0.3,
    )
    answer = llm_response.choices[0].message.content

    return {"answer": answer, "sources": sources}


# --- Demo: Ask 3 example questions ---
demo_questions = [
    "What do customers think about room service?",
    "Are hosts generally responsive and helpful?",
    "What types of properties have the best ratings?",
]

for q in demo_questions:
    print("=" * 80)
    print(f"❓ Question: {q}")
    print("=" * 80)
    result = ask_review_question(q)
    print(f"\n💡 Answer:\n{result['answer']}")
    print(f"\n📋 Sources ({len(result['sources'])} reviews retrieved):")
    for s in result["sources"]:
        print(f"  - [ID {s['review_id']}] {s['property_title']} | "
              f"Rating: {s['rating']} | Score: {s['score']:.4f}")
    print()